# Colab runner
Thin launcher only: **no logic lives here** (it is all in `src/` and `scripts/`). Use a **CPU** runtime (Runtime > Change runtime type).
One-time per person: add your GitHub token as a Colab Secret named `GH_TOKEN` (key icon in the left bar, turn on *Notebook access*).
Run the cells top to bottom; each pipeline stage is its own cell so you can run only yours.

In [ ]:
# ---------- CONFIG (edit) ----------
GITHUB_REPO  = "manahilcpkhan/tomato-robust-ml"
GIT_USER, GIT_EMAIL = "Your Name", "you@example.com"
DATA_VARIANT = "Dataset (raw)"        # folder inside the zip used as data_root (raw photos, we make our own split)
MY_BRANCH    = "a/data"                # a/... (Member A), b/... (Member B), c/... (Member C)
# -----------------------------------

In [ ]:
import glob, os, shutil, subprocess, sys
from google.colab import drive, userdata
drive.mount("/content/drive")

def sh(*cmd, check=True):
    return subprocess.run(list(cmd), check=check)

# --- repo
token = userdata.get("GH_TOKEN")
REPO = "/content/repo"
if not os.path.exists(REPO):
    sh("git", "clone", f"https://{token}@github.com/{GITHUB_REPO}.git", REPO)
else:
    sh("git", "-C", REPO, "pull", "--rebase")
os.chdir(REPO)
sh("git", "config", "user.name", GIT_USER); sh("git", "config", "user.email", GIT_EMAIL)
sh(sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt")

# --- dataset: unzip the zip from Drive to LOCAL disk (reading thousands of files straight from Drive is very slow)
zips = glob.glob("/content/drive/MyDrive/**/Tomato_Leaf_Disease_Dataset_Pakistan.zip", recursive=True)
assert zips, "zip not found in My Drive: check the file name / that Drive is mounted"
ZIP = zips[0]
DRIVE_PROJECT = os.path.dirname(ZIP)
DRIVE_BACKUP = os.path.join(DRIVE_PROJECT, "work_backup")
if not os.path.exists("/content/data/.done"):
    os.makedirs("/content/data", exist_ok=True)
    sh("unzip", "-q", "-o", ZIP, "-d", "/content/data")
    open("/content/data/.done", "w").close()
hits = [d for d, _, _ in os.walk("/content/data") if os.path.basename(d) == DATA_VARIANT]
assert hits, f"no folder named {DATA_VARIANT!r} under /content/data"
DATA = hits[0]

# --- paths for the pipeline: big files on local disk, small results inside the repo
os.environ["TOMATO_DATA_ROOT"]   = DATA
os.environ["TOMATO_WORK_DIR"]    = "/content/work"
os.environ["TOMATO_RESULTS_DIR"] = f"{REPO}/results"
print("data_root:", DATA, "| backup:", DRIVE_BACKUP, "| cpus:", os.cpu_count())

## Restore previous work from Drive
Colab wipes local disk when a session ends. Run this in every NEW session after the first (it copies `work/` back).

In [ ]:
if os.path.exists(DRIVE_BACKUP):
    shutil.copytree(DRIVE_BACKUP, "/content/work", dirs_exist_ok=True)
    print("restored work/ from Drive")
else:
    print("no backup yet (normal on the very first run)")

## v2 RE-RUN ONLY: archive the first run and clear derived files
Run this **once**, right after pulling the new code (cell 2) and before stage 01. It copies the old `results/` to Drive (`archive_v1`), renames the old Drive backup to `work_backup_v1`, and clears local `work/` and `results/` so no stale features or numbers are reused. The old results also stay in git history on branch `a/data`.

In [ ]:
import time
arch = os.path.join(DRIVE_PROJECT, "archive_v1"); os.makedirs(arch, exist_ok=True)
if os.path.isdir("results") and os.listdir("results"):
    shutil.copytree("results", os.path.join(arch, "results_v1"), dirs_exist_ok=True)
if os.path.exists(DRIVE_BACKUP):
    os.rename(DRIVE_BACKUP, DRIVE_BACKUP + "_v1")
shutil.rmtree("/content/work", ignore_errors=True)
for f in glob.glob("results/*"):
    os.remove(f) if os.path.isfile(f) else shutil.rmtree(f)
print("archived v1 -> ", arch, "| work/ and results/ cleared")

## 00 Audit dataset (Member A): class names, counts, near-duplicates
Read the printout. With `--hash` it also clusters near-identical photos (1-3 min) so you can see how many copies exist.

In [ ]:
!python scripts/00_audit_dataset.py --hash

## 01 Prepare data (Member A): grouped train/val/test split + manifest + image cache
Splits by source photo (near-duplicates stay together), 70/15/15 per class, seed from `config.yaml`. Run once, then commit `data/manifest.csv` and `results/grouping_summary.json` via a PR so everyone uses the SAME split.

In [ ]:
!python scripts/01_prepare_data.py

## 02 EDA, segmentation check, duplicates (Member A)

In [ ]:
!python scripts/02_eda_duplicates.py

## 03 Features (Member B)
Benchmark first, then the full extraction.

In [ ]:
!python scripts/03_extract_features.py --benchmark 100

In [ ]:
!python scripts/03_extract_features.py

## 04 Baseline (RF / SVM / LR)  ·  05 Ablation (Member B)

In [ ]:
!python scripts/04_baseline.py

In [ ]:
!python scripts/05_ablation.py

## 06 Robustness  ·  07 Importance stability  ·  08 Error analysis  ·  09 Figures (Member C)

In [ ]:
!python scripts/06_robustness.py

In [ ]:
!python scripts/07_importance_stability.py

In [ ]:
!python scripts/08_error_analysis.py

In [ ]:
!python scripts/09_figures.py

## Step 1 extras (after stages 00-09 have been re-run)
**10** metadata-only classifier (does image size/compression reveal the class?) · **11** 5 repeated grouped splits (mean ± std). Both are fast.

In [ ]:
!python scripts/10_source_shortcut_check.py

In [ ]:
!python scripts/11_repeated_splits.py --seeds 5

## Segmentation comparison: HSV (main run) vs GrabCut vs no mask
Same manifest/split and same cached images; only the segmentation setting changes. Each variant gets its own work/results folders, so nothing is overwritten. GrabCut is roughly 10x slower (expect ~10 min for features + ~30 min for robustness); run **after** the main pipeline is done and backed up.

In [ ]:
import subprocess
VARIANTS = {"none": "configs/seg_none.yaml", "grabcut": "configs/seg_grabcut.yaml"}
for name, cfgfile in VARIANTS.items():
    wd = f"/content/work_{name}"; rd = f"{REPO}/results_seg_{name}"
    os.makedirs(wd, exist_ok=True); os.makedirs(rd, exist_ok=True)
    for f in glob.glob("/content/work/images_cache*"):
        shutil.copy(f, wd)                       # same cached images -> only segmentation differs
    env = dict(os.environ, TOMATO_CONFIG=cfgfile, TOMATO_WORK_DIR=wd, TOMATO_RESULTS_DIR=rd)
    for s in ["03_extract_features.py", "04_baseline.py", "06_robustness.py"]:
        print(f"===== {name}: {s}", flush=True)
        subprocess.run([sys.executable, f"scripts/{s}"], env=env, check=True)

## Save: back up `work/` to Drive and push small results to your branch
Then open a pull request on GitHub. Never push to `main` from the notebook.

In [ ]:
os.makedirs(DRIVE_BACKUP, exist_ok=True)
if os.path.exists("/content/work"):
    shutil.copytree("/content/work", DRIVE_BACKUP, dirs_exist_ok=True)
sh("git", "checkout", "-B", MY_BRANCH)
sh("git", "add", "results", "data/manifest.csv", "config.yaml", "configs", check=False)
for d in glob.glob("results_seg_*"):
    sh("git", "add", d, check=False)
sh("git", "commit", "-m", "results: update from notebook run", check=False)
sh("git", "push", "-u", "origin", MY_BRANCH)